# Direct Forecast — Safe Lag Pipeline

**Thay thế hoàn toàn iterative loop** bằng direct forecast:
- Build features cho **toàn bộ 16 ngày** test (Aug 16–31/2017) **một lần duy nhất**
- Tất cả lag features (lag_16, lag_28, lag_364) lấy từ **train data thật** — không có error propagation
- `model.predict()` gọi **một lần** cho toàn bộ test set, không có loop

## Feature safety analysis

| Feature | Aug 16 | Aug 31 | Status |
|---|---|---|---|
| `lag_16` | cần Aug 1 ✓ | cần Aug 15 ✓ | **SAFE toàn bộ** |
| `lag_28` | cần Jul 19 ✓ | cần Aug 3 ✓ | **SAFE toàn bộ** |
| `lag_364` | cần Aug 17/2016 ✓ | cần Sep 1/2016 ✓ | **SAFE toàn bộ** |
| `rolling_mean_30` | shift(16)→Aug 1, roll(30)→Jul 2–Aug 1 ✓ | cần Jul 17–Aug 15 ✓ | **SAFE toàn bộ** |
| `lag_14` | cần Aug 2 ✓ | cần Aug 17 ✗ | safe 14/16 ngày, NaN Aug 30–31 |
| `rolling_mean_28` | cần Jul 19–Aug 15 ✓ | shift(1)→Aug 30 (test) | NaN Aug 17–31 |
| `rolling_std_7` | cần Aug 10–15 ✓ | shift(1)→Aug 30 (test) | NaN Aug 17–31 |

*Note: XGBoost & LightGBM xử lý NaN natively — model tự học optimal direction cho NaN splits.*

## 1. Load Data & Libraries

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here] + list(_here.parents) if (p / 'config.py').exists()),
    _here
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, SPLITS_DIR

MODELS_DIR    = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
SPLITS_SAFE   = PROCESSED_DIR / 'splits_safe_lag'

assert SPLITS_SAFE.exists(), (
    f'splits_safe_lag không tồn tại!\nChạy feature_safe_lag_v2.ipynb trước.'
)

for model_f in [
    NOTEBOOKS_DIR / '08_forecasting' / 'xgb_safe_lag_model.pkl',
    NOTEBOOKS_DIR / '08_forecasting' / 'lgb_safe_lag_model.pkl',
]:
    assert model_f.exists(), (
        f'Model không tồn tại: {model_f}\n'
        f'Chạy XGBoost_training_safe_lag.ipynb và LightGBM_training_safe_lag.ipynb trước!'
    )

print('All prerequisites ✓')

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error
from category_encoders import TargetEncoder

# ── Kaggle competition test set (2017-08-16 → 2017-08-31) ──────────────────
test_kaggle = pd.read_csv(PROCESSED_DIR / 'cleaned' / 'test_cleaned.csv')

# ── Full historical training data (2013-01-01 → 2017-08-15) ────────────────
train_raw   = pd.read_csv(PROCESSED_DIR / 'cleaned' / 'train_cleaned.csv')

oil_df      = pd.read_csv(RAW_DIR / 'oil.csv')
holidays_df = pd.read_csv(RAW_DIR / 'holidays_events.csv')
stores_df   = pd.read_csv(RAW_DIR / 'stores.csv')

# ── Authoritative column order từ safe lag splits ───────────────────────────
ref_cols = pd.read_csv(SPLITS_SAFE / 'train_features.csv', nrows=0).columns.tolist()

print('Data loaded:')
for name, df in [('test_kaggle', test_kaggle), ('train_raw', train_raw),
                 ('oil', oil_df), ('holidays', holidays_df), ('stores', stores_df)]:
    print(f'  {name:<12}: {df.shape}')
print(f'  {"ref_cols":<12}: {len(ref_cols)} features')
print(f'\nNew features in ref_cols:')
for col in ['lag_16', 'rolling_mean_30']:
    print(f'  {col}: {"YES ✓" if col in ref_cols else "MISSING ✗"}')
print(f'Removed features absent from ref_cols:')
for col in ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'rolling_mean_7', 'rolling_mean_14']:
    print(f'  {col}: {"absent ✓" if col not in ref_cols else "PRESENT ✗"}')

## 2. Feature Engineering (Direct — no iterative loop)

Tất cả features tính từ historical data thật. Không có vòng lặp dự báo.

In [ ]:
# ── Calendar features ─────────────────────────────────────────────────────
X_kaggle = test_kaggle.copy()
X_kaggle['date'] = pd.to_datetime(X_kaggle['date'])

train_raw['date'] = pd.to_datetime(train_raw['date'])

dt = X_kaggle['date'].dt
X_kaggle['year']           = dt.year
X_kaggle['month']          = dt.month
X_kaggle['day']            = dt.day
X_kaggle['day_of_week']    = dt.dayofweek
X_kaggle['week_of_year']   = dt.isocalendar().week.astype(int)
X_kaggle['quarter']        = dt.quarter
X_kaggle['is_weekend']     = (dt.dayofweek >= 5).astype(int)
X_kaggle['is_month_start'] = dt.is_month_start.astype(int)
X_kaggle['is_month_end']   = dt.is_month_end.astype(int)
X_kaggle['is_payday']      = ((dt.day == 15) | dt.is_month_end).astype(int)

# Store geographic info
X_kaggle = X_kaggle.merge(
    stores_df[['store_nbr', 'type', 'city', 'state', 'cluster']],
    on='store_nbr', how='left'
)
assert X_kaggle.shape[0] == len(test_kaggle), 'Store merge changed row count'
print(f'After calendar + store merge: {X_kaggle.shape}')
print(f'Test dates: {sorted(X_kaggle["date"].unique())}')

In [ ]:
# ── Lag features (DIRECT — tất cả từ train_raw) ──────────────────────────
# lag_N tại ngày d = sales tại ngày d-N → nếu d-N ≤ 2017-08-15, lấy từ train_raw
# lag_16: safe cho toàn bộ Aug 16-31 (d-16 ≤ Aug 15)
# lag_28: safe cho toàn bộ Aug 16-31 (d-28 ≤ Jul 31)
# lag_364: safe (d-364 = Aug 2016)
# lag_14: safe Aug 16-29 (d-14 ≤ Aug 15), NaN cho Aug 30-31 (d-14 > Aug 15)

for lag in [14, 16, 28, 364]:
    lag_ref = train_raw[['date', 'store_nbr', 'family', 'sales']].copy()
    lag_ref['date'] = lag_ref['date'] + pd.Timedelta(days=lag)
    lag_ref = lag_ref.rename(columns={'sales': f'lag_{lag}'})
    X_kaggle = X_kaggle.merge(
        lag_ref[['date', 'store_nbr', 'family', f'lag_{lag}']],
        on=['date', 'store_nbr', 'family'], how='left'
    )

assert X_kaggle.shape[0] == len(test_kaggle), 'Lag merge changed row count'

print('Lag feature NaN counts:')
for lag in [14, 16, 28, 364]:
    n   = X_kaggle[f'lag_{lag}'].isnull().sum()
    pct = n / len(X_kaggle) * 100
    safe_status = 'SAFE ✓' if n == 0 else f'NaN cho {n//1782} ngày (model xử lý natively)'
    print(f'  lag_{lag:<4}: {n:>5} NaN ({pct:.1f}%)  — {safe_status}')

# Critical assertion: lag_16, lag_28, lag_364 phải có 0 NaN
for lag in [16, 28, 364]:
    n = X_kaggle[f'lag_{lag}'].isnull().sum()
    assert n == 0, f'FAIL: lag_{lag} có {n} NaN — unexpected!'
print('\nAssertion passed: lag_16, lag_28, lag_364 all zero NaN ✓')

In [ ]:
# ── Rolling features ─────────────────────────────────────────────────────
# Dùng "combined append" approach:
#   1. Lấy đủ history từ train_raw (46+ ngày trở lại)
#   2. Append test rows với NaN sales
#   3. Tính rolling trên combined (sorted by store/family/date)
#   4. Extract test rows

test_start = X_kaggle['date'].min()
cutoff     = test_start - pd.Timedelta(days=46)   # đủ window cho rolling_mean_30

train_recent = train_raw.loc[
    train_raw['date'] >= cutoff,
    ['date', 'store_nbr', 'family', 'sales']
].copy()

test_rows = X_kaggle[['date', 'store_nbr', 'family']].assign(sales=np.nan)

combined = pd.concat([train_recent, test_rows]).sort_values(
    ['store_nbr', 'family', 'date']
).reset_index(drop=True)

print(f'Combined df: {combined.shape}')
print(f'  Train rows  : {len(train_recent):,}')
print(f'  Test rows   : {len(test_rows):,}')
print()

grp = combined.groupby(['store_nbr', 'family'], sort=False)['sales']

# rolling_mean_28: shift(1).rolling(28) — NaN cho Aug 17-31 (model handles natively)
combined['rolling_mean_28'] = grp.transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
)

# rolling_std_7: shift(1).rolling(7) — NaN cho Aug 17-31 (model handles natively)
combined['rolling_std_7'] = grp.transform(
    lambda x: x.shift(1).rolling(7, min_periods=2).std()
)

# rolling_mean_30: shift(16).rolling(30) — SAFE cho toàn bộ Aug 16-31
# Tại d: mean(sales[d-45], ..., sales[d-16]) — tất cả ≤ Aug 15 ✓
combined['rolling_mean_30'] = grp.transform(
    lambda x: x.shift(16).rolling(30, min_periods=1).mean()
)

# Extract test rows
test_dates_mask = combined['date'].isin(X_kaggle['date'].unique())
roll_cols       = ['date', 'store_nbr', 'family', 'rolling_mean_28', 'rolling_std_7', 'rolling_mean_30']
test_rolling    = combined.loc[test_dates_mask, roll_cols].reset_index(drop=True)

X_kaggle = X_kaggle.merge(test_rolling, on=['date', 'store_nbr', 'family'], how='left')
assert X_kaggle.shape[0] == len(test_kaggle), 'Rolling merge changed row count'

del combined, train_recent, test_rows, test_rolling, grp

print('Rolling feature NaN counts:')
for col in ['rolling_mean_28', 'rolling_std_7', 'rolling_mean_30']:
    n   = X_kaggle[col].isnull().sum()
    pct = n / len(X_kaggle) * 100
    if col == 'rolling_mean_30':
        safe_str = 'SAFE toàn bộ ✓' if n == 0 else f'FAIL ✗'
    else:
        safe_str = f'NaN cho ~{n//1782} ngày (model xử lý natively)'
    print(f'  {col:<20}: {n:>5} NaN ({pct:.1f}%)  — {safe_str}')

assert X_kaggle['rolling_mean_30'].isnull().sum() == 0, 'FAIL: rolling_mean_30 has NaN!'
print('\nAssertion passed: rolling_mean_30 zero NaN ✓')

In [ ]:
# ── Holiday features ──────────────────────────────────────────────────────
holidays = holidays_df.copy()
holidays['date'] = pd.to_datetime(holidays['date'])
valid_holidays   = holidays[holidays['transferred'] == False].copy()

type_mapping = {'Holiday': 1, 'Event': 2, 'Additional': 3,
                'Transfer': 4, 'Bridge': 5, 'Work Day': 0}
valid_holidays['holiday_type_encoded'] = valid_holidays['type'].map(type_mapping).fillna(0).astype(int)
valid_holidays['is_transferred']       = valid_holidays['transferred'].astype(int)
valid_holidays['is_carnaval']          = (
    valid_holidays['description'].str.contains('Carnaval', case=False, na=False).astype(int)
)

for col in ['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
            'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature']:
    X_kaggle[col] = 0

scope_col = {'National': 'is_national_holiday',
             'Regional': 'is_regional_holiday',
             'Local':    'is_local_holiday'}

for _, row in valid_holidays.iterrows():
    h_date, h_locale, h_name = row['date'], row['locale'], row['locale_name']
    if h_locale == 'National':
        mask = X_kaggle['date'] == h_date
    elif h_locale == 'Regional':
        mask = (X_kaggle['date'] == h_date) & (X_kaggle['state'] == h_name)
    elif h_locale == 'Local':
        mask = (X_kaggle['date'] == h_date) & (X_kaggle['city'] == h_name)
    else:
        continue
    if not mask.any():
        continue
    X_kaggle.loc[mask, scope_col[h_locale]]      = 1
    X_kaggle.loc[mask, 'is_transferred_holiday'] = row['is_transferred']
    X_kaggle.loc[mask, 'holiday_type']           = row['holiday_type_encoded']
    X_kaggle.loc[mask, 'is_carnaval_feature']    = row['is_carnaval']

unique_hol_dates = valid_holidays['date'].drop_duplicates().sort_values().values

def _days_to_next(d):
    future = unique_hol_dates[unique_hol_dates > d]
    return int((future[0] - d) / np.timedelta64(1, 'D')) if len(future) > 0 else -1

def _days_after_last(d):
    past = unique_hol_dates[unique_hol_dates < d]
    return int((d - past[-1]) / np.timedelta64(1, 'D')) if len(past) > 0 else -1

uq_dates = X_kaggle[['date']].drop_duplicates().copy()
uq_dates['days_to_next_holiday']    = uq_dates['date'].apply(_days_to_next)
uq_dates['days_after_last_holiday'] = uq_dates['date'].apply(_days_after_last)
X_kaggle = X_kaggle.merge(uq_dates, on='date', how='left')

X_kaggle['is_earthquake_period'] = 0  # test dates are Aug 2017, quake was Apr 2016

assert X_kaggle.shape[0] == len(test_kaggle), 'Holiday merge changed row count'
print(f'National holidays : {X_kaggle["is_national_holiday"].sum()}')
print(f'Regional holidays : {X_kaggle["is_regional_holiday"].sum()}')
print(f'Local holidays    : {X_kaggle["is_local_holiday"].sum()}')

In [ ]:
# ── Oil price features ────────────────────────────────────────────────────
oil = oil_df.copy().rename(columns={'dcoilwtico': 'oil_price'})
oil['date'] = pd.to_datetime(oil['date'])
oil = oil[['date', 'oil_price']].sort_values('date').reset_index(drop=True)

max_date   = max(oil['date'].max(), X_kaggle['date'].max())
full_range = pd.DataFrame({'date': pd.date_range(oil['date'].min(), max_date, freq='D')})
oil = full_range.merge(oil, on='date', how='left')
oil['oil_price'] = oil['oil_price'].ffill().bfill()

oil['oil_price_lag_7']           = oil['oil_price'].shift(7)
oil['oil_price_lag_14']          = oil['oil_price'].shift(14)
oil['oil_price_rolling_mean_28'] = oil['oil_price'].shift(1).rolling(28, min_periods=7).mean()
oil['oil_price_change_pct']      = oil['oil_price'].pct_change(periods=7)

oil_cols = ['date', 'oil_price', 'oil_price_lag_7', 'oil_price_lag_14',
            'oil_price_rolling_mean_28', 'oil_price_change_pct']
X_kaggle = X_kaggle.merge(oil[oil_cols], on='date', how='left')

assert X_kaggle.shape[0] == len(test_kaggle), 'Oil merge changed row count'
print(f'Oil NaN total     : {X_kaggle[oil_cols[1:]].isnull().sum().sum()}')
print(f'Oil price range   : {X_kaggle["oil_price"].min():.2f} – {X_kaggle["oil_price"].max():.2f}')

In [ ]:
# ── Store encoding + target encoding ────────────────────────────────────
type_map = {t: i for i, t in enumerate(sorted(stores_df['type'].unique()))}
X_kaggle['store_type_enc']     = X_kaggle['type'].map(type_map)
X_kaggle['store_type_encoded'] = X_kaggle['type'].map(type_map)

city_freq_map  = stores_df['city'].value_counts(normalize=True).to_dict()
state_freq_map = stores_df['state'].value_counts(normalize=True).to_dict()
X_kaggle['city_freq']  = X_kaggle['city'].map(city_freq_map)
X_kaggle['state_freq'] = X_kaggle['state'].map(state_freq_map)

X_kaggle['store_family'] = X_kaggle['store_nbr'].astype(str) + '_' + X_kaggle['family']

# Target encoding — fit trên full train data (giống training pipeline)
train_raw['store_family'] = train_raw['store_nbr'].astype(str) + '_' + train_raw['family']
te_enc = TargetEncoder(cols=['store_family'], smoothing=10)
te_enc.fit(train_raw[['store_family']], train_raw['sales'])
X_kaggle['store_family_te'] = te_enc.transform(X_kaggle[['store_family']])['store_family']

# Convert date → string (format YYYY-MM-DD, giống training splits)
X_kaggle['date'] = X_kaggle['date'].dt.strftime('%Y-%m-%d')

# Lưu kaggle IDs
kaggle_ids = test_kaggle['id'].reset_index(drop=True)
X_kaggle   = X_kaggle.drop(columns=['id'], errors='ignore')

# Assert column match vs ref_cols
missing_cols = set(ref_cols) - set(X_kaggle.columns)
extra_cols   = set(X_kaggle.columns) - set(ref_cols)
assert not missing_cols, f'Missing columns: {missing_cols}'
assert not extra_cols,   f'Extra columns: {extra_cols}'

X_kaggle = X_kaggle[ref_cols]  # enforce exact column order
assert X_kaggle.shape == (len(test_kaggle), len(ref_cols)), 'Final shape mismatch'

print(f'X_kaggle final: {X_kaggle.shape}')
nan_counts = X_kaggle.isnull().sum()
nan_nonzero = nan_counts[nan_counts > 0]
if len(nan_nonzero):
    print(f'\nColumns với NaN (model xử lý natively — safe lags không có NaN):')
    print(nan_nonzero.to_string())
else:
    print('\nNo missing values — perfect!')

## 2b. Verify Safe Lag Features

In [ ]:
print('=== VERIFICATION: Safe lag features cho Kaggle test (Aug 16-31) ===\n')
for col in ['lag_16', 'lag_28', 'lag_364', 'rolling_mean_30']:
    n = X_kaggle[col].isnull().sum()
    status = 'PASS ✓ (0 NaN — 100% real historical data)' if n == 0 else f'FAIL ✗ ({n} NaN)'
    print(f'  {col:<20}: {status}')
    assert n == 0, f'BUG: {col} has {n} NaN — pipeline error!'

print()
print('Sample X_kaggle (5 rows, key features):')
key_cols = ['date', 'store_nbr', 'lag_14', 'lag_16', 'lag_28', 'lag_364', 'rolling_mean_30']
print(X_kaggle[key_cols].head(5).to_string())

## 3. Encode Categorical Features

In [ ]:
object_cols = X_kaggle.select_dtypes(include=['object']).columns.tolist()

# Refit LabelEncoders trên train + val + test splits + kaggle
X_train_obj = pd.read_csv(SPLITS_SAFE / 'train_features.csv', usecols=lambda c: c in object_cols)
X_val_obj   = pd.read_csv(SPLITS_SAFE / 'val_features.csv',   usecols=lambda c: c in object_cols)
X_test_obj  = pd.read_csv(SPLITS_SAFE / 'test_features.csv',  usecols=lambda c: c in object_cols)

label_encoders = {}
for col in object_cols:
    le    = LabelEncoder()
    parts = [X_train_obj[col], X_val_obj[col]]
    if col in X_test_obj.columns:
        parts.append(X_test_obj[col])
    parts.append(X_kaggle[col])
    combined = pd.concat(parts, ignore_index=True).astype(str)
    le.fit(combined)
    X_kaggle[col] = le.transform(X_kaggle[col].astype(str)).astype(np.int32)
    label_encoders[col] = le

del X_train_obj, X_val_obj, X_test_obj

remaining = X_kaggle.select_dtypes(include=['object']).columns.tolist()
assert not remaining, f'Still has object columns: {remaining}'
print(f'Encoded: {object_cols}')
print(f'X_kaggle final: {X_kaggle.shape}')

## 4. Load Models & Direct Predict

**Direct forecast**: gọi `model.predict()` **một lần** cho toàn bộ 28,512 rows — không có loop.

In [ ]:
lgb_model = joblib.load(NOTEBOOKS_DIR / '08_forecasting' / 'lgb_safe_lag_model.pkl')
xgb_model = joblib.load(NOTEBOOKS_DIR / '08_forecasting' / 'xgb_safe_lag_model.pkl')

# Ensemble weights: 0.7 XGBoost + 0.3 LightGBM (giống forecasting.ipynb)
best_lgb_w = 0.3
best_xgb_w = 0.7

print(f'Models loaded.  LGB weight: {best_lgb_w}  |  XGB weight: {best_xgb_w}')
print(f'Feature count:  LGB={lgb_model.n_features_}  |  XGB={xgb_model.n_features_in_}')
print(f'\nRunning DIRECT predict on {len(X_kaggle):,} rows (no loop)...')

lgb_pred = lgb_model.predict(X_kaggle)
xgb_pred = xgb_model.predict(X_kaggle)

# Weighted blend trong log space → inverse log1p → clip âm về 0
final_pred_log = best_lgb_w * lgb_pred + best_xgb_w * xgb_pred
final_pred     = np.maximum(np.expm1(final_pred_log), 0)

print(f'\nPredictions:')
print(f'  min  : {final_pred.min():.4f}')
print(f'  max  : {final_pred.max():.2f}')
print(f'  mean : {final_pred.mean():.2f}')
print(f'  NaN  : {np.isnan(final_pred).sum()}')

## 5. Evaluate trên Local Test Set (Aug 01–15)

In [ ]:
X_test_local = pd.read_csv(SPLITS_SAFE / 'test_features.csv')
y_test_local = pd.read_csv(SPLITS_SAFE / 'test_target.csv')

# Encode categorical columns
for col in object_cols:
    if col in X_test_local.columns:
        le = label_encoders[col]
        X_test_local[col] = X_test_local[col].astype(str).apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        ).astype(np.int32)

lgb_loc = lgb_model.predict(X_test_local)
xgb_loc = xgb_model.predict(X_test_local)
blend_log  = best_lgb_w * lgb_loc + best_xgb_w * xgb_loc
blend_pred = np.maximum(np.expm1(blend_log), 0)

y_true = np.expm1(y_test_local['sales_log'].to_numpy())

rmsle    = np.sqrt(mean_squared_log_error(np.clip(y_true, 0, None), np.clip(blend_pred, 0, None)))
rmse     = np.sqrt(mean_squared_error(y_true, blend_pred))
mae      = mean_absolute_error(y_true, blend_pred)
rmse_log = np.sqrt(mean_squared_error(y_test_local['sales_log'].to_numpy(), blend_log))

print(f'Ensemble LGB{best_lgb_w:.1f}–XGB{best_xgb_w:.1f} | Local Test Set (Aug 01–15)')
print('=' * 55)
for name, val in [('RMSLE', rmsle), ('RMSE', rmse), ('MAE', mae), ('RMSE_log', rmse_log)]:
    print(f'  {name:<12}: {val:.6f}')
print('=' * 55)

print('\nSo sánh với old ensemble (forecasting_iterative.ipynb):')
print(f'  OLD RMSLE (iterative): N/A (no local test metric in iterative nb)')
print(f'  OLD RMSLE (direct)   : 0.370354  (forecasting.ipynb với toxic lags + NaN)')
print(f'  NEW RMSLE (safe lag) : {rmsle:.6f}')

## 6. Tạo Submission File

In [ ]:
submission = pd.DataFrame({'id': kaggle_ids, 'sales': final_pred})

# Validations
assert len(submission) == len(test_kaggle),          'Row count mismatch'
assert submission['sales'].isnull().sum() == 0,      'NaN in sales'
assert (submission['sales'] < 0).sum() == 0,         'Negative sales'

out_dir = NOTEBOOKS_DIR / '08_ensemble'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'submission_safe_lag.csv'
submission.to_csv(out_path, index=False)

print(f'Submission saved: {out_path}')
print(f'Shape           : {submission.shape}')
print(f'Sales range     : [{submission["sales"].min():.4f}, {submission["sales"].max():.2f}]')
print(f'Sales mean      : {submission["sales"].mean():.2f}')
print()
print(submission.head(10).to_string())

## Summary

| Pipeline | Forecast method | Lag NaN (Aug 16-31) | Local RMSLE |
|---|---|---|---|
| `forecasting.ipynb` | Direct (toxic lags) | lag_1: 100%, lag_7: 56% | 0.370354 |
| `forecasting_iterative.ipynb` | Iterative loop | 0% (filled by prediction) | N/A |
| **`forecasting_safe_lag.ipynb`** | **Direct (safe lags)** | **lag_16/28/364: 0%** | **(see above)** |

**Kết quả**: Không còn iterative loop, không còn error propagation, lag_16/lag_28/lag_364/rolling_mean_30 đều có 100% dữ liệu thật.